# Limpieza, integración y puerta de calidad (E1) · Clima para eventos al aire libre en Colima

**Introducción.** Este notebook explica, paso a paso y con los datos reales del repositorio, qué hace el pipeline Bronce → Plata → Oro. El pipeline que corre solo después de cada captura está en `pipeline/` (`python -m pipeline.ejecutar`). Aquí se usan **las mismas funciones** para mostrar cada decisión: valores faltantes, duplicados, integración de las dos fuentes, puerta de calidad con cuarentena, atípicos, normalización, discretización, jerarquías y reducción.

El notebook lo ejecuta GitHub Actions (flujo `analisis`), así que los números que se ven abajo salen de la última corrida con los datos de Bronce de ese momento.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RAIZ = Path.cwd()
while not (RAIZ / "pipeline").exists():  # el notebook vive en docs/E1; se busca la raiz del repositorio
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from pipeline import bronce_a_plata as bp
from pipeline import plata_a_oro as po
from pipeline import puerta_calidad as pc

pd.set_option("display.width", 140)
print("Repositorio:", RAIZ.name)

## 1. Capa Bronce: los datos tal como llegaron

Bronce guarda cada respuesta de las APIs sin modificarla, con su fecha de captura. Hay dos fuentes: **Open-Meteo** (serie principal) y **NASA POWER** (segunda fuente para comparar). Como la captura repite ventanas de días para no dejar huecos, el mismo municipio-día puede venir varias veces.

In [ ]:
om = bp.leer_openmeteo("openmeteo_archive", RAIZ)
nasa = bp.leer_nasa(RAIZ)
print(f"Open-Meteo: {len(om):,} filas leídas · NASA POWER: {len(nasa):,} filas leídas")
om.head(3)

## 2. Valores faltantes: diagnóstico del mecanismo

**En simple:** hay que preguntarse *por qué* falta un dato, porque de eso depende qué hacer con él. Si falta al azar (MCAR), quitarlo no sesga nada; si falta por algo que sí se observa, por ejemplo la fecha (MAR), se puede explicar y controlar; si falta por el valor mismo que no vemos (MNAR), por ejemplo que el sensor falle justo cuando llueve mucho, es el caso más peligroso.

**Versión formal:** MCAR (*missing completely at random*): la probabilidad de faltar no depende de ninguna variable. MAR (*missing at random*): depende de variables observadas. MNAR (*missing not at random*): depende del valor no observado.

NASA marca con `-999` los días que todavía no procesa; el lector los convierte a vacío. La tabla compara qué tanto falta en los días recientes contra los anteriores.

In [ ]:
print("Faltantes en Open-Meteo (variables principales):",
      int(om[["tmax_c", "tmin_c", "lluvia_mm"]].isna().sum().sum()))
print("Días de NASA con lluvia faltante (crudo, antes de quitar duplicados):", int(nasa["nasa_lluvia_mm"].isna().sum()))

nasa_u, _ = bp.quitar_duplicados(nasa)
ultimo = nasa_u["fecha"].max()
nasa_u["antiguedad"] = (ultimo - nasa_u["fecha"]).dt.days.map(lambda d: "últimos 30 días" if d <= 30 else "anteriores")
(nasa_u.groupby("antiguedad")["nasa_lluvia_mm"].apply(lambda s: s.isna().mean() * 100)
       .round(2).rename("% faltante").to_frame())

**Decisión:** el faltante se concentra en los días recientes, o sea, depende de la fecha (una variable observada): es **MAR**. NASA publica con retraso y las capturas posteriores traen el dato. **No se rellena**: la serie principal (Open-Meteo) está completa y NASA solo se usa para comparar; inventar valores de comparación agregaría error. Alternativas descartadas: media o mediana (aplana la lluvia, que es muy asimétrica), interpolación temporal o KNN (no hacen falta porque la serie principal no tiene huecos).

## 3. Duplicados (*entity resolution* dentro de cada fuente)

La clave de un registro es **municipio + fecha**. Si un día llegó en varias capturas, se conserva la **más reciente**, porque los datos consolidados pueden corregirse unos días después.

In [ ]:
om_u, n_om = bp.quitar_duplicados(om)
nasa_u, n_nasa = bp.quitar_duplicados(nasa)
print(f"Open-Meteo: {n_om:,} duplicados quitados -> {len(om_u):,} registros únicos")
print(f"NASA POWER: {n_nasa:,} duplicados quitados -> {len(nasa_u):,} registros únicos")

## 4. Integración de las dos fuentes

Conflictos que se resolvieron al unirlas en un solo modelo:

| Conflicto | Open-Meteo | NASA POWER | Resolución |
|---|---|---|---|
| Nombres de variables | `precipitation_sum`, `temperature_2m_max`… | `PRECTOTCORR`, `T2M_MAX`… | Nombres comunes en español (`lluvia_mm`, `tmax_c`); NASA con prefijo `nasa_` |
| Formato de fecha | `AAAA-MM-DD` | `AAAAMMDD` | Ambas a tipo fecha |
| Unidad del viento | km/h | m/s | NASA × 3.6 |
| Valor faltante | vacío | `-999` | `-999` → vacío |

La unión es por municipio y fecha, y **Open-Meteo manda** (unión izquierda): cada día de la serie principal queda una sola vez, con o sin pareja de NASA.

In [ ]:
integrada = bp.integrar(om_u, nasa_u)
print(f"Registros integrados: {len(integrada):,} · sin pareja de NASA: {int(integrada['nasa_lluvia_mm'].isna().sum()):,}")
integrada[["dif_lluvia_mm", "dif_tmax_c"]].describe().round(2)

Las diferencias entre fuentes (`dif_*`) no se corrigen: se conservan como información de **credibilidad** para la evaluación ISO/IEC 25012 de E2.

## 5. Puerta de calidad: cuarentena con motivo

Cada regla de negocio o rango físico se revisa registro por registro. Lo que falla **no se borra**: va a `data/cuarentena/cuarentena.csv` con el motivo. Después, el esquema de pandera valida tipos, rangos y unicidad de toda la capa Plata.

In [ ]:
for nombre, _ in pc.REGLAS:
    print("-", nombre)
aprobados, cuarentena = pc.aplicar_reglas(integrada)
print(f"\nAprobados: {len(aprobados):,} · en cuarentena: {len(cuarentena):,}")

Con los datos reales la cuarentena puede salir vacía. Para demostrar que la puerta sí funciona, se prueba con tres registros alterados a propósito (solo en memoria, no se guardan):

In [ ]:
prueba = integrada.head(3).copy()
prueba.loc[prueba.index[0], "tmin_c"] = prueba["tmax_c"].iloc[0] + 5   # mínima mayor que la máxima
prueba.loc[prueba.index[1], "lluvia_mm"] = -4.0                       # lluvia negativa
prueba.loc[prueba.index[2], "viento_max_kmh"] = 900.0                 # viento imposible
_, rechazados = pc.aplicar_reglas(prueba)
rechazados[["municipio", "fecha", "motivo"]]

## 6. Atípicos: se marcan, no se borran

**En simple:** un día con 150 mm de lluvia en Colima es raro, pero puede ser real (un huracán). Para el cliente, ese día es justamente el más importante. Por eso los atípicos se **marcan** en columnas `atipico_*` y se conservan; solo lo físicamente imposible va a cuarentena (paso 5).

**Versión formal:** Z-score robusto de Iglewicz y Hoaglin, `0.6745 · (x − mediana) / MAD`, calculado por municipio y mes del año (para no marcar como atípico todo el mes de septiembre), con umbral 3.5. Para la lluvia se usan solo los días con 1 mm o más y en escala `log(1 + mm)`, porque su distribución es muy asimétrica.

In [ ]:
plata = pc.marcar_atipicos(aprobados)
plata = po.discretizar(plata)
print({c: int(plata[c].sum()) for c in plata.columns if c.startswith("atipico_")})
plata[plata["atipico_lluvia_mm"]].nlargest(5, "lluvia_mm")[["municipio", "fecha", "lluvia_mm"]]

In [ ]:
col = plata[plata["municipio"] == "colima"]
fig, eje = plt.subplots(figsize=(11, 3.5))
eje.plot(col["fecha"], col["lluvia_mm"], linewidth=0.6, color="gray", label="lluvia diaria")
marcados = col[col["atipico_lluvia_mm"]]
eje.scatter(marcados["fecha"], marcados["lluvia_mm"], color="red", s=18, label="marcado como atípico")
eje.set_ylabel("mm")
eje.set_title("Municipio de Colima: lluvia diaria y atípicos marcados")
eje.legend()
plt.show()

## 7. Discretización y jerarquías conceptuales

La lluvia diaria se convierte en categorías que el cliente entiende (un "día de lluvia" es de 1 mm o más, criterio de la OMM). También se construyen dos jerarquías: **día → mes → temporada → año** y **municipio → zona → estado**.

In [ ]:
print(plata["categoria_lluvia"].value_counts().reindex(po.ETIQUETAS_LLUVIA))
jerarquia = plata.groupby(["zona", "municipio"]).size().rename("días").reset_index()
jerarquia.insert(0, "estado", "Colima")
jerarquia

## 8. Reducción: de días a meses

La capa Oro agrega por municipio y mes. Se gana una serie manejable para pronosticar y comparar; se pierde el detalle dentro del mes (qué día exacto llovió y con qué intensidad), aunque se conservan los días de lluvia y la lluvia máxima diaria de cada mes.

In [ ]:
oro = po.a_mensual(plata, por=("municipio",))
print(f"{len(plata):,} registros diarios -> {len(oro):,} registros municipio-mes: "
      f"reducción del {(1 - len(oro) / len(plata)) * 100:.1f} %")
oro.tail(3)

## 9. Normalización: min-max contra z-score

**En simple:** para comparar lluvia (cientos de mm) con temperatura (decenas de °C) hay que ponerlas en la misma escala. Min-max las lleva a 0–1; z-score las expresa en "cuántas desviaciones estándar se alejan del promedio".

**Versión formal:** min-max `(x − mín) / (máx − mín)`; z-score `(x − media) / desviación estándar`.

In [ ]:
completos = oro[oro["mes_completo"]]
norm = po.normalizar(completos, ["lluvia_total_mm", "tmax_media_c"])
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.5))
ejes[0].hist(norm["lluvia_total_mm_minmax"], bins=30)
ejes[0].set_title("Lluvia mensual · min-max")
ejes[1].hist(norm["lluvia_total_mm_z"], bins=30)
ejes[1].set_title("Lluvia mensual · z-score")
plt.show()
norm[["lluvia_total_mm_minmax", "lluvia_total_mm_z", "tmax_media_c_minmax", "tmax_media_c_z"]].describe().round(2)

**Decisión:** con min-max, un solo mes extremo fija el máximo y la mayoría de los meses quedan apretados cerca de 0. El z-score conserva la forma y deja ver qué tan extremo es cada mes. Para el PCA de E3 se usará **z-score**.

## Conclusiones

Los datos viajan de Bronce a Oro sin intervención y sin perder rastro: los duplicados se resuelven conservando la captura más reciente, los faltantes de NASA se explican (MAR) y no se inventan, lo imposible va a cuarentena con su motivo, lo raro se marca sin borrarse y cada fila de Plata conserva su archivo de Bronce y su hora de captura (linaje). Los números de cada corrida automática quedan en `docs/E1/reporte_pipeline.md`.